In [ ]:
# 필수 패키지 설치
!pip install -q langgraph langchain-openai langchain-chroma langchain-huggingface sentence-transformers langchain-community

In [ ]:
import os
import getpass
from typing import List, TypedDict
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

# OpenAI API Key 설정
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 임베딩 모델 & DB 로드
print("--- 임베딩 모델 및 DB 로드 중... ---")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="./chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

In [ ]:
# 상태(State) 정의
class GraphState(TypedDict):
    summary: str                    # 입력: 상담 요약
    search_query: str               # 중간산출: 검색 키워드
    documents: List[Document]       # 중간산출: 검색된 문서
    organized_guide: str            # 출력: 신입 상담원용 가이드

In [ ]:
# [Node 1] 키워드 추출 (gpt-5-nano)
def query_analyzer_node(state: GraphState):
    summary = state["summary"]
    print(f"\n📊 [분석 중] 상담 요약 내용: {summary}")

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=0,
        max_tokens=100
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", """약관 검색용 키워드를 3~5개 추출하세요.
예시:
입력: 고객이 이사로 인해 인터넷을 이전 설치하려고 하는데 비용이 발생하는지 문의함
출력: 인터넷 이전 설치비 댁내 이전 출동비 면제 조건"""),
        ("user", "{summary}")
    ])

    response = (prompt | llm).invoke({"summary": summary})
    refined_query = response.content

    print(f"🔑 [키워드 추출 완료] -> '{refined_query}'")
    return {"search_query": refined_query}

In [ ]:
# [Node 2] 문서 검색
def retriever_node(state: GraphState):
    query = state["search_query"]
    print(f"📚 [DB 검색 수행] 키워드: {query}")
    results = vectorstore.similarity_search(query, k=3)
    return {"documents": results}

In [ ]:
# [Node 3] 정보 재구성 (gpt-5-mini)
def organize_info_node(state: GraphState):
    summary = state["summary"]
    documents = state["documents"]
    print(f"📝 [정보 재구성 중] 검색된 문서 {len(documents)}개 분석 중...")
    
    docs_text = "\n\n".join([
        f"[문서 {i+1}] {doc.metadata.get('source', '약관').split('/')[-1]} (p.{doc.metadata.get('page', 0)+1})\n{doc.page_content}"
        for i, doc in enumerate(documents)
    ])
    
    # 프롬프트 버전: VERSION_1 (간결형)
    prompt_v1 = """당신은 신입 상담원을 돕는 교육 담당자입니다.
아래 약관 정보를 분석하여 신입 상담원이 고객에게 안내할 내용을 정리하세요.

[상담 상황]
{summary}

[검색된 약관]
{docs_text}

다음 형식으로 작성하세요:
1. 핵심 내용 (1-2문장)
2. 고객 안내 멘트
3. 참고 약관"""

    # 프롬프트 버전: VERSION_2 (상세형) - 기본
    prompt_v2 = """당신은 신입 상담원을 위한 약관 안내 전문가입니다.
검색된 약관을 분석하여 신입 상담원이 바로 활용할 수 있도록 정리하세요.

[상담 상황]
{summary}

[검색된 약관]
{docs_text}

다음 형식으로 작성하세요:

1️⃣ **핵심 내용 요약**
   - 이 상담에서 가장 중요한 약관 내용을 2-3문장으로 요약

2️⃣ **고객 안내 스크립트**
   - 신입 상담원이 그대로 읽을 수 있는 멘트 작성
   - 친절하고 이해하기 쉬운 표현 사용

3️⃣ **주의사항 / 예외조건**
   - 고객 상황에 따라 달라질 수 있는 조건들
   - 확인해야 할 추가 정보

4️⃣ **관련 약관 출처**
   - 어떤 약관 문서의 몇 페이지인지 명시"""

    # 프롬프트 버전: VERSION_3 (단계별 가이드형)
    prompt_v3 = """당신은 신입 상담원의 멘토입니다.
약관을 분석하여 단계별 상담 가이드를 작성하세요.

[상담 상황]
{summary}

[검색된 약관]
{docs_text}

다음 형식으로 작성하세요:

🔍 **상황 분석**
   고객이 원하는 것: 
   적용 가능한 약관:

📞 **상담 진행 순서**
   STEP 1: [첫 번째로 확인할 사항]
   STEP 2: [두 번째로 안내할 내용]
   STEP 3: [마무리 안내]

⚠️ **주의사항**
   - 주의 1:
   - 주의 2:

📋 **약관 출처**"""

    # 프롬프트 선택 (환경변수: PROMPT_VERSION = v1/v2/v3, 기본: v2)
    prompt_version = os.environ.get("PROMPT_VERSION", "v2")
    if prompt_version == "v1":
        system_prompt = prompt_v1
    elif prompt_version == "v3":
        system_prompt = prompt_v3
    else:
        system_prompt = prompt_v2
    
    llm = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0.3,
        max_tokens=800
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "상담 상황: {summary}\n\n약관 내용:\n{docs_text}")
    ])
    
    response = (prompt | llm).invoke({
        "summary": summary,
        "docs_text": docs_text
    })
    
    organized_guide = response.content
    print(f"✅ [정보 재구성 완료]")
    return {"organized_guide": organized_guide}

In [ ]:
# 그래프 연결
workflow = StateGraph(GraphState)
workflow.add_node("analyze_query", query_analyzer_node)
workflow.add_node("retrieve_docs", retriever_node)
workflow.add_node("organize_info", organize_info_node)

workflow.add_edge(START, "analyze_query")
workflow.add_edge("analyze_query", "retrieve_docs")
workflow.add_edge("retrieve_docs", "organize_info")
workflow.add_edge("organize_info", END)

app = workflow.compile()

In [ ]:
# 실행 함수
def run_junior_consulting_assistant(conversation_summary):
    inputs = {"summary": conversation_summary}
    result = app.invoke(inputs)
    
    print("\n" + "="*80)
    print(f"💬 입력된 상담 요약:\n{conversation_summary}")
    print("="*80)
    print(f"\n🔑 추출된 검색 키워드:\n{result['search_query']}")
    print("-"*80)
    print(f"\n📚 검색된 약관 문서: {len(result['documents'])}개")
    print("-"*80)
    print(f"\n📋 신입 상담원을 위한 안내 가이드:\n")
    print(result['organized_guide'])
    print("="*80)
    return result

In [ ]:
# 테스트 실행
summary_input = "고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다."
result = run_junior_consulting_assistant(summary_input)

## 추가 테스트

프롬프트 버전 변경:
```python
# VERSION 1 (간결형)
os.environ["PROMPT_VERSION"] = "v1"
result = run_junior_consulting_assistant("고객이 인터넷 속도 저하 문제로 환불을 요청합니다.")

# VERSION 3 (단계별)
os.environ["PROMPT_VERSION"] = "v3"
result = run_junior_consulting_assistant("고객이 이사를 가는데 인터넷 이전 절차와 비용에 대해 문의합니다.")
```

**상세 문서**: `DOCUMENTATION.md` - 모델 비용, 하이퍼파라미터, 예상 결과 등